<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Inference costs

**[Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/)** · Deep Learning and Transformers Explained · Module 7, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the memory of a model from its parameter count, look inside the KV cache, time generation with and without the cache and with batching, try int8 quantization, and count what one training step adds.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 270 MB for the model). No account, no graphics card. About 3 GB of free memory for the last section (free Colab has about 12 GB). |
| **You should know** | SmolLM2-135M and the generation loop, from [Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled; Hugging Face transformers 5.19.0 |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M07-L02-inference-costs/inference-costs-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. It also installs Hugging Face `transformers` for the small pretrained model SmolLM2-135M (about 270 MB to download later).

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
%pip install -q --disable-pip-version-check "transformers>=5,<6" safetensors
import transformers; print("transformers", transformers.__version__)
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: load SmolLM2-135M

Run the next cell. It downloads the small pretrained model **SmolLM2-135M** from Hugging Face and loads it. The model is free and open (licence Apache-2.0), and it needs no account.

- The first run downloads about **270 MB**. The makers store the weights in the 16-bit format bfloat16, 2 bytes per weight. The download can take a minute; later runs use the saved copy.
- The cell loads the weights as **float32**, 4 bytes per weight, so the model takes about **540 MB** of memory. The lessons' numbers were measured in float32, and every CPU computes float32 well. [Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) explains the difference.
- `revision=` asks for one exact version of the model files, so your model is the course's model, even if the makers update theirs.

You should see a progress bar, then `parameters: 134515008`.

If the cell shows an error:

- `OSError` or `Connection`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `No module named 'transformers'`: run the first setup cell again.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-135M"
REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"  # the exact version of the files that the course used
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.float32)
model.eval()                                   # inference: switch off training-only behaviour
PROMPT = ("Larkfield is an online shop for home and garden products. "
          "A customer wrote: my parcel is late. The support team")
print("parameters:", sum(p.numel() for p in model.parameters()))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Memory of the weights: count × bytes

Each parameter is one number. A float32 number takes 4 bytes, bfloat16 2 bytes, int8 1 byte and int4 half a byte. So the memory of the weights is the parameter count times the bytes per number.

> **Predict.** SmolLM2-135M has 134,515,008 parameters. How many megabytes (millions of bytes) in float32? Then run the cell.

In [ ]:
n = sum(p.numel() for p in model.parameters())
for name, bytes_per_number in [("float32", 4), ("bfloat16", 2), ("int8", 1), ("int4", 0.5)]:
    print(f"{name:9} {n * bytes_per_number / 1e6:7.1f} MB")
loaded = sum(p.numel() * p.element_size() for p in model.parameters())
print("the loaded model:", loaded, "bytes")

> **Check.** You should see `float32 538.1 MB`, `bfloat16 269.0 MB`, `int8 134.5 MB`, `int4 67.3 MB` and `the loaded model: 538060032 bytes`: the arithmetic and the real tensors agree.

> **Your turn.** A model has 1.7 billion parameters. Compute the memory of its weights in bfloat16, in gigabytes (1 GB = 1,000,000,000 bytes). Put the number in `gb_bf16`, then run the check cell.

In [ ]:
gb_bf16 = ...  # 1.7 billion parameters x 2 bytes, in GB

In [ ]:
expect('gb_bf16', gb_bf16, 3.4, tolerance=0.01)

## 2. Activations: the numbers in between

Activations are the numbers that a forward pass computes on its way: the hidden states after each block and the logits at the end. They exist only during the pass, and they grow with the number of tokens.

> **Predict.** The logits for a prompt of 25 tokens have the shape (1, 25, 49152). How many bytes is that in float32? Then run the cell.

In [ ]:
inputs = tokenizer(PROMPT, return_tensors="pt")
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)
show_shape("one hidden", out.hidden_states[1])
show_shape("logits", out.logits)
print("hidden state bytes:", out.hidden_states[1].numel() * 4)
print("logits bytes:      ", out.logits.numel() * 4)

> **Check.** You should see `one hidden shape (1, 25, 576)`, `logits shape (1, 25, 49152)`, `hidden state bytes: 57600` and `logits bytes: 4915200`: the logits of 25 tokens take almost 5 MB, because every position gets 49,152 scores.

## 3. Inside the KV cache

In attention, each token's query is compared with the **keys** of all earlier tokens, and the result mixes their **values**. The keys and values of earlier tokens never change, so `generate()` keeps them: the **KV cache**. The next cell runs the prompt once with `use_cache=True` (the **prefill**) and looks inside.

> **Predict.** The model has 30 blocks and 3 key-value heads of 64 numbers. What shape will the keys of one block have for 25 tokens?

In [ ]:
with torch.no_grad():
    out = model(**inputs, use_cache=True)
cache = out.past_key_values
print("blocks in the cache:", len(cache.layers))
show_shape("keys", cache.layers[0].keys)
show_shape("values", cache.layers[0].values)
cache_bytes = sum(l.keys.numel() * 4 + l.values.numel() * 4 for l in cache.layers)
print("cache bytes for 25 tokens:", cache_bytes, "| per token:", cache_bytes // 25)

> **Check.** You should see `blocks in the cache: 30`, `keys shape (1, 3, 25, 64)`, `values shape (1, 3, 25, 64)` and `cache bytes for 25 tokens: 1152000 | per token: 46080`.

The shape reads: 1 text, 3 key-value heads, 25 tokens, 64 numbers. Per token: 2 (keys and values) × 30 blocks × 3 heads × 64 numbers × 4 bytes = 46,080 bytes.

Now one **decode** step: feed only the newest token, with the cache.

In [ ]:
next_id = out.logits[0, -1].argmax().reshape(1, 1)
with torch.no_grad():
    step = model(input_ids=next_id, past_key_values=cache, use_cache=True)
show_shape("input", next_id)
show_shape("logits", step.logits)
show_shape("keys now", cache.layers[0].keys)

> **Check.** You should see `input shape (1, 1)`, `logits shape (1, 1, 49152)` and `keys now shape (1, 3, 26, 64)`: one token in, one row of scores out, and the cache grew by one token.

> **Your turn.** SmolLM2 can read up to 8,192 tokens. Compute the bytes of the KV cache in float32 for one text of 8,192 tokens. Put the number in `cache_8k`, then run the check cell.

In [ ]:
cache_8k = ...  # bytes per token x 8,192

In [ ]:
expect('cache_8k', cache_8k, 377487360)

## 4. Time it: with and without the cache

The next cell generates 40 tokens twice: with the cache, and with `use_cache=False`, which runs the whole text again at every step. The outputs are the same; only the time differs. **Your times depend on your computer**: the lesson's numbers come from a quiet Mac mini, measured five times.

> **Predict.** How much slower will generation be without the cache?

In [ ]:
import time

def timed_generate(**settings):
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id, **settings)
    return time.perf_counter() - start, out

timed_generate(use_cache=True)                       # warm-up: the first call is slower
t_cache, a = timed_generate(use_cache=True)
t_none, b = timed_generate(use_cache=False)
print(f"with the cache    {t_cache:.2f} s")
print(f"without the cache {t_none:.2f} s")
print("same tokens:", torch.equal(a, b))

> **Check.** You should see `same tokens: True` and a longer time without the cache. The course's computer took 0.46 s with the cache and 1.01 s without. With longer texts the gap grows, because each step without the cache reads every earlier token again.

## 5. Batching: eight texts at once

A **batch** of prompts goes through the model together, so each weight is read once for all of them. Prompts of different lengths need **padding**: filler tokens that make every row the same length. For generation, the padding goes on the left, so that every text ends at the same place. SmolLM2 has no padding token, so the cell reuses its end-of-text token.

In [ ]:
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token            # SmolLM2 has no padding token: reuse the end token
batch = tokenizer([PROMPT] * 8, return_tensors="pt", padding=True)
one = tokenizer(PROMPT, return_tensors="pt")
show_shape("batch", batch["input_ids"])

def run(inp):
    with torch.no_grad():
        return model.generate(**inp, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

start = time.perf_counter()
for _ in range(8):
    run(one)
t_one = time.perf_counter() - start
start = time.perf_counter()
run(batch)
t_batch = time.perf_counter() - start
print(f"8 texts one by one {t_one:.2f} s | 8 texts in one batch {t_batch:.2f} s")

> **Check.** You should see `batch shape (8, 25)` and a shorter time for the batch. The course's computer took 1.92 s one by one and 0.47 s as one batch. Each text in a batch waits for the whole batch, and the batch needs eight caches.

## 6. Quantization: smaller, but faster? The same?

**Dynamic int8 quantization** stores the weights of every `Linear` layer as 8-bit integers and converts them during the computation. The cell copies the model, quantizes the copy, and compares the next-token probabilities and the greedy text.

If the cell stops with `AttributeError` or `ModuleNotFoundError` for `torch.ao.quantization`, your PyTorch is newer than the course's and has moved this function to the separate `torchao` package. Skip this section: the lesson page shows the course's result.

> **Predict.** Will the int8 copy give the same top tokens with the same probabilities as the float32 model?

In [ ]:
import copy, platform, warnings
engines = torch.backends.quantized.supported_engines
arm = platform.machine().lower() in ("arm64", "aarch64")
torch.backends.quantized.engine = "qnnpack" if arm and "qnnpack" in engines else ("x86" if "x86" in engines else "fbgemm")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")              # PyTorch says this older API will move to the torchao package
    q_model = torch.ao.quantization.quantize_dynamic(copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8)

def top5(m):
    with torch.no_grad():
        p = torch.softmax(m(**one).logits[0, -1], dim=-1)
    t = torch.topk(p, 5)
    return [(tokenizer.decode([int(i)]), round(v.item(), 3)) for v, i in zip(t.values, t.indices)]

print("engine:", torch.backends.quantized.engine)
print("float32:", top5(model))
print("int8:   ", top5(q_model))
with torch.no_grad():
    q_out = q_model.generate(**one, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("int8 text:", repr(tokenizer.decode(q_out[0, one["input_ids"].shape[1]:])))

> **Check.** The float32 line starts with `(' is', 0.107)`. The int8 line is different. On the course's Mac (engine `qnnpack`), `' is'` rose to 0.195 and the text became `' is not answering my emails. …'`. On a computer with another engine (`x86` on Colab), the int8 numbers differ again.

Quantization saves memory. It does not always save time: on the course's CPU, 40 tokens took 0.60 s in int8 against 0.46 s in float32. And it changes the outputs, so you must measure quality again after you quantize.

In [ ]:
del q_model  # free the memory of the copy

## 7. Training needs much more memory

Inference needs the weights, the activations of one pass and the cache. Training also needs a **gradient** for every weight, and the optimizer Adam keeps **two more numbers** per weight (a running average of the gradients and of their squares). The next cell makes one real training step on SmolLM2 and counts the bytes.

> **Warning.** This cell needs about 3 GB of memory, and it changes the model's weights. Run it last. To use the model again afterwards, run the setup cell that loads the model.

> **Predict.** The weights take 538 MB. How much will weights + gradients + Adam's state take?

In [ ]:
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
batch = tokenizer(PROMPT, return_tensors="pt")
loss = model(**batch, labels=batch["input_ids"]).loss   # next-token prediction on the prompt
loss.backward()
optimizer.step()

mb = lambda tensors: sum(t.numel() * t.element_size() for t in tensors) / 1e6
weights = mb(model.parameters())
gradients = mb(p.grad for p in model.parameters())
adam = mb(v for state in optimizer.state.values() for k, v in state.items() if k in ("exp_avg", "exp_avg_sq"))
print(f"weights {weights:.0f} MB | gradients {gradients:.0f} MB | Adam {adam:.0f} MB | total {weights + gradients + adam:.0f} MB")
_ = model.eval()

> **Check.** You should see `weights 538 MB | gradients 538 MB | Adam 1076 MB | total 2152 MB`: four times the weights, before the activations that training must also keep for the backward pass.

> **Your turn.** Tomás wants Larkfield to train its own model of 70 billion parameters. Compute weights + gradients + Adam's state in float32 (4 copies of 4 bytes per parameter), in gigabytes. Put the number in `gb_70b`, then run the check cell.

In [ ]:
gb_70b = ...  # 70 billion parameters x 4 copies x 4 bytes, in GB

In [ ]:
expect('gb_70b', gb_70b, 1120.0, tolerance=0.01)

That is about 1,120 GB, before any activations: 70 times the 16 GB of a good laptop. Large models are trained on many large GPUs at once. Ordinary hardware can run small models for inference, and can fine-tune small ones.

## Next

You can now estimate the memory of weights, activations and the KV cache, and you have seen what the cache, batching and int8 quantization do to time and outputs. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) has the knowledge checks. The next lesson, [Save and inspect results](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/), saves a model and records an experiment so that another person can repeat it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Save and inspect results](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). SmolLM2-135M is by Hugging Face, under the Apache-2.0 licence.